<a href="https://colab.research.google.com/github/rolandnwonodi/GAADS-Optimization-for-Fracture-length/blob/main/Fraclengthphysic_informed_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# -------------------------------
# 1 Load dataset
# -------------------------------

# Replace with your file name
data = pd.read_excel("fracture_dataset.xlsx")

# Required columns in dataset
# mu, Pnet, ti, qi, K, E, Lf

mu = data["mu"].values
Pnet = data["Pnet"].values
ti = data["ti"].values
qi = data["qi"].values
K = data["K"].values
E = data["E"].values
Lf = data["Lf"].values


# -------------------------------
# 2 Construct dimensionless terms
# -------------------------------

X1 = mu / (Pnet * ti)

X2 = (qi * ti) / (K**1.5)

X3 = E / Pnet

# Target variable in log space
Y = np.log(Lf / np.sqrt(K))


# -------------------------------
# 3 Define model
# -------------------------------

def model(params, X1, X2, X3):

    logR, d, e, c = params

    prediction = (
        logR
        + d * np.log(X1)
        + c * np.log(X2)
        + e * np.log(X3)
    )

    return prediction


# -------------------------------
# 4 Define loss function
# -------------------------------

def loss_function(params, X1, X2, X3, Y):

    prediction = model(params, X1, X2, X3)

    error = Y - prediction

    return np.sum(error**2)


# -------------------------------
# 5 Initial Guess (Your Excel Model)
# -------------------------------

initial_guess = (
    np.log(0.009),   # ln(R)
    -0.6382,         # d
    0.75,            # e
    0.39             # c
)


# -------------------------------
# 6 Run optimization
# -------------------------------

result = minimize(
    loss_function,
    initial_guess,
    args=(X1, X2, X3, Y),
    method='L-BFGS-B'
)

logR, d, e, c = result.x

R = np.exp(logR)

print("\nOptimized Model Coefficients\n")

print("R =", R)
print("d =", d)
print("c =", c)
print("e =", e)


# -------------------------------
# 7 Predict fracture length
# -------------------------------

Lf_pred = (
    R
    * (mu/(Pnet*ti))**d
    * ((qi*ti)/(K**1.5))**c
    * (E/Pnet)**e
    * np.sqrt(K)
)


# -------------------------------
# 8 Model accuracy
# -------------------------------

RMSE = np.sqrt(np.mean((Lf - Lf_pred)**2))

R2 = 1 - np.sum((Lf - Lf_pred)**2) / np.sum((Lf - np.mean(Lf))**2)

print("\nModel Performance")

print("RMSE =", RMSE)
print("R² =", R2)


# -------------------------------
# 9 Parity Plot
# -------------------------------

plt.figure(figsize=(6,6))

plt.scatter(Lf, Lf_pred)

plt.plot([min(Lf), max(Lf)], [min(Lf), max(Lf)])

plt.xlabel("True Fracture Length")
plt.ylabel("Predicted Fracture Length")

plt.title("Model Parity Plot")

plt.grid()

plt.show()


# -------------------------------
# 10 Print final equation
# -------------------------------

print("\nFinal Trained Equation:\n")

print(
    f"L_f = {R:.5f} "
    f"(μ/(P_net t_i))^{d:.3f} "
    f"((q_i t_i)/K^1.5)^{c:.3f} "
    f"(E/P_net)^{e:.3f} "
    f"K^(1/2)"
)